In [2]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from google.genai import types
from google import genai
from pydantic import BaseModel, Field
import json

In [4]:
client = genai.Client()
model = "gemini-3-flash-preview"

res = client.models.generate_content(
  model=model,
  contents="what is the capital of France?"
)

res.text

'The capital of France is **Paris**.'

In [5]:
# lets create a research agent which can take question and then search the web and then answer the question based on the search results.
from utils import get_search_results

query = input("Enter query you want to search : ")

In [8]:
question_generater_prompt = """
You are a helpful assistant for generating questions based on the asked query. You will be given a query and you have to generate 5 questions based on the query which can be used to search the web for getting more information about the query. The questions should be related to the query and should be specific enough to get relevant search results.
For example, if the query is "What is blockchain?", then the generated questions can be:
1. What are the key features of blockchain technology?
2. How does blockchain work?
3. What are the advantages of using blockchain?
4. What are the use cases of blockchain?
5. What are the challenges of blockchain technology?

Now, generate 5 questions based on the following query: 
<query>
{query}
</query>
"""

class Question(BaseModel):
    query: str = Field(..., description="The query for which questions need to be generated")

class QuestionGeneratorOutput(BaseModel):
    questions: list[Question] = Field(..., description="A list of 5 questions generated based on the query")

def generate_questions(input : str) -> QuestionGeneratorOutput:
    prompt = question_generater_prompt.format(query=input)
    res = client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=QuestionGeneratorOutput
        )
    )
    questions = json.loads(res.text).get("questions", [])
    return questions

questions = generate_questions(query)

In [9]:
from pprint import pprint
pprint(questions)

[{'query': 'What are the fundamental principles of blockchain technology and '
           'its core components?'},
 {'query': 'How can blockchain improve data privacy and security in machine '
           'learning models?'},
 {'query': 'What are the benefits of using blockchain for decentralized AI '
           'training and inference?'},
 {'query': 'In what ways does blockchain provide transparency and traceability '
           'for AI decision-making processes?'},
 {'query': 'What are some real-world examples or use cases where blockchain '
           'and AI/ML are integrated?'}]


In [ ]:
from ddgs import DDGS
import trafilatura
from pprint import pprint

def search_question(question):
    """
    This function takes a question as input, performs a web search using DuckDuckGo, and retrieves the text content from the top search results. It returns a list of dictionaries containing the title, URL, and extracted text for each search result.
    """
    search_results = []
    results = DDGS().text(question, max_results=5)
    for result in results:
        if result["href"]:
            downloaded = trafilatura.fetch_url(result["href"])
            if downloaded:
                text = trafilatura.extract(downloaded)
                search_results.append({
                    "title": result["title"],
                    "href": result["href"],
                    "text": text,
                })
    return search_results

def get_search_results(questions):
    """
    This function takes a list of questions as input, performs a web search for each question using the `search_question` function, and aggregates the search results. It returns a list of dictionaries containing the question and its corresponding search results.
    """
    search_results = []
    for question in questions:
        results = search_question(question['query'])
        search_results.append({
            "question": question,
            "results": results,
        })
    return search_results

In [11]:
res = get_search_results(questions)

In [12]:
res

[{'question': {'query': 'What are the fundamental principles of blockchain technology and its core components?'},
  'results': [{'title': 'Components of Blockchain Network - GeeksforGeeks',
    'href': 'https://www.geeksforgeeks.org/solidity/components-of-blockchain-network/',
    'text': 'Blockchain networks have various interdependent components that work together to ensure secure, transparent, and efficient data transactions. Key elements include nodes, which validate and relay transactions; a decentralized ledger that records all activity; and consensus mechanisms that maintain the integrity of the network. Additionally, cryptographic techniques and smart contracts enhance security and automate processes. This article discusses the components of the Blockchain Network in detail.\nCore Components of Blockchain Networks\nThe core components of blockchain networks are essential for their operation and functionality. Each component plays a critical role in maintaining the integrity, se

In [16]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")
total_tokens = 0

for item in res:
    for result in item.get("results", []):
        text = str(result.get("text", ""))
        total_tokens += len(tokenizer.encode(text))

print(f"Total tokens used in search results: {total_tokens}")

Total tokens used in search results: 36467


In [17]:
# now lets create a model which will create a ans based on all of this content 
create_answer_prompt = """You are a helpful assistant for answering questions based on the search results. You will be given a question and a list of search results. You have to read through the search results and then generate a concise and accurate answer to the question based on the information available in the search results. The answer should be based on the information available in the search results and should not include any information that is not present in the search results. The answer should be concise and to the point, and should not include any unnecessary information. The ans should be easy to understand 
Here is the question and the search results:
<question>
{question}
</question>

<search_results>
{search_results}
</search_results>
"""

class AnswerGeneratorOutput(BaseModel):
    answer: str = Field(..., description="The answer generated based on the search results")

def generate_answer(question, search_results) -> AnswerGeneratorOutput:
    prompt = create_answer_prompt.format(question=question, search_results=search_results)
    res = client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=AnswerGeneratorOutput
        )
    )
    answer = json.loads(res.text).get("answer", "")
    return answer

# now creating a proper markdown ans with sources cited for each section / key statement.
create_markdown_answer_prompt = """You will be given a question, an answer, and a set of original documents.

Your task is to rewrite the answer in markdown and add source citations for every important line, section, or factual statement so the reader can clearly see which document each part came from.

Rules:
- Cite sources inline after each sentence or bullet using the document title or a document number like [Doc 1].
- If a section uses multiple documents, cite all relevant documents.
- If information is only supported by one document, cite only that document.
- Do not add any information that is not supported by the documents.
- If a part of the answer cannot be traced to any document, omit it.
- Make the result concise, readable, and well-structured with headings and bullets where helpful.

Here is the question, the answer, and the original documents:

<question>
{question}
</question>

<answer>
{answer}
</answer>

<documents>
{documents}
</documents>

Return the final answer in markdown with inline citations for each section or sentence."""

class MarkdownAnswerOutput(BaseModel):
    markdown_answer: str = Field(..., description="The answer rewritten in markdown format with source citations")

def generate_markdown_answer(question, answer, documents) -> MarkdownAnswerOutput:
    prompt = create_markdown_answer_prompt.format(question=question, answer=answer, documents=documents)
    res = client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=MarkdownAnswerOutput
        )
    )
    markdown_answer = json.loads(res.text).get("markdown_answer", "")
    return markdown_answer

In [18]:
def generate_final_answer(question, search_results):
    answer = generate_answer(question, search_results)
    markdown_answer = generate_markdown_answer(question, answer, search_results)
    return markdown_answer

In [19]:
final_ans = generate_final_answer(query, res)

In [22]:
from IPython.display import display, Markdown

display(Markdown(final_ans))

### What is Blockchain?
Blockchain is a decentralized, immutable, and transparent distributed ledger technology that records transactions and data across a network of computers, known as nodes [Components of Blockchain Network - GeeksforGeeks, Blockchain Technology: Core Mechanisms, Evolution, and Future Implementation Challenges]. It utilizes cryptography and consensus mechanisms, such as Proof of Work or Proof of Stake, to ensure data integrity without the need for a central authority [Components of Blockchain Network - GeeksforGeeks, Blockchain Architecture Explained: Comprehensive Guide]. This technology provides a shared "single view of the truth," fostering trust among participants through permanent, encrypted records [What is blockchain and artificial intelligence (AI)? - IBM].

### Using Blockchain in AI and Machine Learning (AIML)
Blockchain enhances AIML by improving security, transparency, and decentralization through the following methods:

*   **Data Integrity and Provenance:** Blockchain provides a tamper-proof record of training datasets and model updates [How Blockchain Improves AI Transparency and Trust]. This helps address the AI "black box" problem by creating an immutable audit trail, making AI decisions traceable and interpretable for users and regulators [What is blockchain and artificial intelligence (AI)? - IBM, Artificial Intelligence and Blockchain: The Definitive Guide | SmartDev].
*   **Privacy and Security:** Decentralized storage and advanced encryption protect sensitive datasets by eliminating single points of failure [The Impact of Blockchain on Data Privacy in AI Systems - Inery]. Zero-Knowledge Proofs (zkML) allow for verifiable inference, where a model's output can be mathematically proven accurate without exposing the underlying private data or model weights [Decentralized AI: Training and Verifiable Inference on Ethereum].
*   **Federated Learning:** Blockchain coordinates decentralized training where models learn from data stored across dispersed nodes [AI in Blockchain in 2026: Key Use Cases - Blockchain Council]. This allows for "collaborative learning" where the global model is updated without the raw, sensitive data ever leaving its original location [Blockchain And AI: Innovative Ways They Can Work Together].
*   **Autonomous AI Agents:** AI agents can be equipped with digital wallets and smart accounts, enabling them to independently execute transactions and interact with smart contracts on-chain [AI in Blockchain in 2026: Key Use Cases - Blockchain Council]. These "agentic" systems can perform economic work, such as paying for compute resources or managing assets, without human intervention [Decentralized AI: Training and Verifiable Inference on Ethereum, Blockchain and generative AI: Fueling innovation within the digital economy | AWS Startups].
*   **Tokenization and Ownership:** Data and models can be tokenized as Non-Fungible Tokens (NFTs) to clarify ownership and licensing rights [AI in Blockchain in 2026: Key Use Cases - Blockchain Council]. This allows contributors to be fairly compensated for their data or model usage through programmable token economies [Blockchain And AI: Innovative Ways They Can Work Together].
*   **Operational Optimization:** AI can be used to improve the blockchain itself by optimizing energy consumption during mining, detecting fraudulent transaction patterns in real-time, and auditing smart contracts for security vulnerabilities [Artificial Intelligence and Blockchain: The Definitive Guide | SmartDev, Machine Learning in Blockchain for AI Engineers and Blockchain Developers].